#基于 MindNLP 的 Wav2Vec2 音频意图识别

## 项目简介
本项目基于 **MindSpore 2.7.0** 和 **MindNLP** 框架，使用预训练的 `facebook/wav2vec2-base` 模型在 **MInDS-14** 数据集上进行微调，实现英语语音意图分类任务。

## 实验环境
* **硬件平台**: Huawei Ascend 910 (NPU)
* **运行模式**: PYNATIVE_MODE (动态图模式)
* **关键策略**: 
    1. **定长输入**: 统一填充至 5秒 (80000采样点)。
    2. **手动解码**: 使用 `librosa` 替代默认解码器，提升底层兼容性。
    3. **梯度屏蔽**: 冻结非浮点型参数 (Int64)，防止优化器报错。

### 1. 环境初始化与依赖配置
配置 HuggingFace 镜像源以加速模型下载，并设置 MindSpore 运行上下文为 Ascend NPU。

In [2]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import numpy as np
import librosa
import mindspore as ms
from datasets import load_dataset, Audio
from mindnlp.transformers import (
    AutoFeatureExtractor, 
    AutoModelForAudioClassification, 
    TrainingArguments, 
    Trainer
)

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="Ascend")

[WARNING] ME(2670341:281473801904160,MainProcess):2026-01-30-12:50:44.499.000 [mindspore/context.py:1412] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


### 2. 数据集加载与划分
加载 MInDS-14 (en-US) 数据集，并按 **8:2** 的比例划分为训练集和验证集。同时构建标签与 ID 的映射字典。

In [3]:
print(">>> Loading Dataset...")
try:
    # 尝试加载 MInDS-14 数据集 (en-US)
    minds = load_dataset("PolyAI/minds14", name="en-US", split="train")
except:
    # 网络波动时的重试机制
    minds = load_dataset("PolyAI/minds14", name="en-US", split="train")

# 划分验证集 (20% 用于验证)
minds = minds.train_test_split(test_size=0.2)

# 构建标签映射字典
labels = minds["train"].features["intent_class"].names
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}

print(f"数据集加载完成，类别数量: {len(labels)}")

>>> Loading Dataset...
数据集加载完成，类别数量: 14


### 3. 数据预处理 (Feature Extraction)
为了适应 Wav2Vec2 模型输入要求，我们执行以下关键操作：
1. **手动读取**: 使用 `librosa` 读取音频文件或字节流，规避系统底层依赖问题。
2. **统一长度**: 将所有音频重采样至 16kHz，并填充/截断至 **80000** 采样点 (5秒)。

In [4]:
model_id = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

# 音频最大长度: 4-5秒 (64000-80000采样点)
# 经验值设为 80000 (5秒) 以覆盖完整指令
MAX_DURATION = 80000 

def preprocess_function(examples):
    """
    读取音频 -> 重采样至16k -> 填充/截断至固定长度
    """
    audio_arrays = []
    for x in examples["audio"]:
        try:
            # 使用 librosa 手动读取，规避 datasets 底层解码依赖问题
            if x.get('bytes'):
                import io
                y, _ = librosa.load(io.BytesIO(x['bytes']), sr=16000)
            else:
                y, _ = librosa.load(x['path'], sr=16000)
            
            # 异常处理：空音频补全
            if len(y) == 0: y = np.zeros(16000)
            audio_arrays.append(y)
        except:
            audio_arrays.append(np.zeros(16000))
    
    # 使用 FeatureExtractor 进行标准化处理
    inputs = feature_extractor(
        audio_arrays, 
        sampling_rate=16000, 
        max_length=MAX_DURATION,
        truncation=True, 
        padding="max_length",
    )
    return inputs

# 移除原始 audio 列，设置为 decode=False 以获取原始字节流
minds = minds.cast_column("audio", Audio(decode=False))

print(">>> Preprocessing Data...")
encoded_minds = minds.map(
    preprocess_function, 
    remove_columns=["audio", "path", "transcription", "english_transcription", "lang_id"], 
    batched=True
)
encoded_minds = encoded_minds.rename_column("intent_class", "labels")

/home/zdai/miniconda3/envs/QA/lib/python3.11/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


>>> Preprocessing Data...


Map: 100%|██████████| 113/113 [00:00<00:00, 215.08 examples/s]


### 4. 模型加载与参数兼容性补丁
加载预训练模型，并应用 **Int64 梯度屏蔽补丁**。这是为了解决 MindSpore 优化器在处理位置编码等整数型参数时可能出现的梯度计算错误。

In [ ]:
print(f">>> Loading Model: {model_id}")
model = AutoModelForAudioClassification.from_pretrained(
    model_id, 
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
)

# 冻结 Int64/Int32 参数梯度
# 说明：Wav2Vec2 的位置编码参数为整数类型，MindSpore 优化器无法对整数求导。
# 必须显式将 requires_grad 设为 False，否则训练会报错。
for p in model.get_parameters():
    if p.dtype in (ms.int32, ms.int64):
        p.requires_grad = False

>>> Loading Model: facebook/wav2vec2-base


/home/zdai/miniconda3/envs/QA/lib/python3.11/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


[MS_ALLOC_CONF]Runtime config:  enable_vmm:True  vmm_align_size:2MB


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 5. 训练参数配置
配置 TrainingArguments，设置学习率、Batch Size 以及 Evaluation 策略。

In [6]:
args = TrainingArguments(
    output_dir="wav2vec2_ckpt",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,             # 经典微调学习率
    per_device_train_batch_size=8,  # 训练 Batch Size
    per_device_eval_batch_size=8,   # 验证 Batch Size
    num_train_epochs=15,            # 正常训练轮次
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    seed=42
)

# 评估指标函数 (引入 evaluate 库)
import evaluate
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

### 6. 模型训练 (Training)
初始化 Trainer 并启动训练。注意此处不传递 `tokenizer` 参数，以避免触发额外的依赖检查。

In [7]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_minds["train"],
    eval_dataset=encoded_minds["test"],
    compute_metrics=compute_metrics,
)

print("\n🚀 开始标准训练...")
trainer.train()


🚀 开始标准训练...


Epoch,Training Loss,Validation Loss,Accuracy
1,2.649500,2.641374,0.079646
2,2.639900,2.617154,0.106195
3,2.483800,2.415874,0.300885
4,2.117700,2.107937,0.566372
5,1.806700,1.876625,0.646018
6,1.418400,1.667981,0.663717
7,1.333700,1.512553,0.672566
8,1.117300,1.409446,0.699115
9,0.909000,1.298606,0.752212
10,0.832500,1.270922,0.716814


TrainOutput(global_step=855, training_loss=1.3736766614412006, metrics={'train_runtime': 1199.8227, 'train_samples_per_second': 5.626, 'train_steps_per_second': 0.713, 'total_flos': 3.0641135868e+17, 'train_loss': 1.3736766614412006, 'epoch': 15.0})

### 7. 模型推理验证 (Inference Demo)
训练完成后，从测试集中随机抽取 3 个样本，进行端到端的推理验证，展示模型预测结果。

In [16]:
print("\n=== 🔎 随机抽样验证 (5条) ===")

import random

# 切换到评估模式
model.set_train(False)

# 获取当前设备名 (适配 Ascend/NPU)
ctx_device = ms.get_context("device_target")
target_device = "npu" if ctx_device == "Ascend" else ctx_device

# 从验证集中随机抽取 5 个样本
test_dataset = encoded_minds["test"]
sample_indices = random.sample(range(len(test_dataset)), 5)

for idx in sample_indices:
    sample = test_dataset[idx]
    
    # 1. 准备输入数据 (增加 Batch 维度 [1, L])
    input_values = np.array(sample["input_values"], dtype=np.float32)
    input_tensor = ms.Tensor([input_values], dtype=ms.float32)
    
    # 2. 移动到设备
    input_tensor = input_tensor.to(target_device)
    
    # 3. 执行推理
    logits = model(input_tensor).logits
    pred_id = np.argmax(logits.asnumpy(), axis=-1)[0]
    
    # 4. 解析标签
    true_label = id2label[str(sample["labels"])]
    pred_label = id2label[str(pred_id)]
    
    # 5. 打印结果
    status = "✅" if true_label == pred_label else "❌"
    print(f"[{status}] 样本 ID: {idx}")
    print(f"   真实意图: {true_label}")
    print(f"   预测意图: {pred_label}")
    print("-" * 40)


=== 🔎 随机抽样验证 (5条) ===
[✅] 样本 ID: 103
   真实意图: atm_limit
   预测意图: atm_limit
----------------------------------------
[✅] 样本 ID: 20
   真实意图: joint_account
   预测意图: joint_account
----------------------------------------
[✅] 样本 ID: 89
   真实意图: freeze
   预测意图: freeze
----------------------------------------
[✅] 样本 ID: 54
   真实意图: pay_bill
   预测意图: pay_bill
----------------------------------------
[✅] 样本 ID: 43
   真实意图: pay_bill
   预测意图: pay_bill
----------------------------------------
